# 3.3 · Backtesting y métricas hidrológicas

**Tiempo estimado:** 15 min.

**Objetivos.**

1. Hacer **backtesting** (walk-forward) de LightGBM sobre el caudal diario.
2. Reutilizar las **métricas hidrológicas** (NSE, KGE, error en pico) del notebook 2.4.
3. Identificar regímenes donde el modelo falla más (estiaje, crecidas).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../sesion1'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold

import utils_datos as ud

plt.rcParams.update({'figure.figsize': (10, 3.4), 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
caudal = ud.cargar_caudal_genil()
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio='2010-01-01', fecha_fin='2020-12-31')

df = pd.DataFrame({'caudal': caudal, 'lluvia': lluvia}).loc['2011':'2020']
df = df.asfreq('D').interpolate('linear', limit=7).dropna()

def features_exog(df):
    out = pd.DataFrame(index=df.index)
    for w in (3, 7, 14, 30):
        out[f'p_acum{w}d'] = df['lluvia'].rolling(w).sum().shift(1)
    idx = out.index
    out['sin_an'] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out['cos_an'] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
    return out.asfreq('D')

X_exog = features_exog(df).dropna().asfreq('D')
y = df['caudal'].loc[X_exog.index].asfreq('D')
print(f'Disponibles: {len(y):,} días   Freq: {y.index.freq}')

## 1 · Backtesting expanding window

Train inicial hasta 2015; cada fold predice 30 días y se reajusta.

In [ ]:
forecaster = ForecasterRecursive(
    regressor=lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                 n_jobs=-1, verbose=-1, random_state=0),
    lags=[1, 2, 3, 7, 14, 30, 90, 365],
)

inicio_eval = pd.Timestamp('2018-01-01')
n_train = (y.index < inicio_eval).sum()
print(f'Días en train inicial: {n_train:,}')

cv = TimeSeriesFold(
    initial_train_size=n_train,
    steps=30,        # horizonte 30 días
    refit=False,     # rápido: ajusta una sola vez. Para más rigor: True (lento)
)

metrica, predicciones = backtesting_forecaster(
    forecaster=forecaster,
    y=y, exog=X_exog,
    cv=cv,
    metric='mean_absolute_error',
    show_progress=False,
)
print(f'MAE global (h=30): {metrica.iloc[0, 0]:.3f}')

In [ ]:
predicciones.head(3)

## 2 · Métricas hidrológicas

Reutilizamos las definiciones del notebook 2.4.

In [ ]:
def nse(o, s):
    o, s = np.asarray(o), np.asarray(s)
    return 1 - np.sum((o - s) ** 2) / np.sum((o - o.mean()) ** 2)

def kge(o, s):
    o, s = np.asarray(o), np.asarray(s)
    r = np.corrcoef(o, s)[0, 1]
    a = s.std() / o.std()
    b = s.mean() / o.mean()
    return 1 - np.sqrt((r - 1) ** 2 + (a - 1) ** 2 + (b - 1) ** 2)

def pbias(o, s):
    return 100 * (np.asarray(o).sum() - np.asarray(s).sum()) / np.asarray(o).sum()

obs = y.loc[predicciones.index].values
sim = predicciones['pred'].values
print(f'NSE  = {nse(obs, sim):.3f}')
print(f'KGE  = {kge(obs, sim):.3f}')
print(f'PBIAS = {pbias(obs, sim):+.1f}%   (negativo = sobreestima)')
print(f'Error pico = {sim.max() - obs.max():+.2f} m³/s  (max obs={obs.max():.2f})')

## 3 · Diagnóstico por régimen

In [ ]:
df_diag = pd.DataFrame({'obs': obs, 'sim': sim, 'err': sim - obs}, index=predicciones.index)
df_diag['mes'] = df_diag.index.month
# Bins por cuartiles (robusto a periodos sin crecida absoluta)
df_diag['regimen'] = pd.qcut(df_diag['obs'], q=4, labels=['estiaje', 'bajo', 'medio', 'alto'])

print('Error medio por régimen (cuartiles):')
print(df_diag.groupby('regimen', observed=True)[['err']].agg(['mean', 'std', 'count']).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(obs, sim, s=4, alpha=0.4, color='#2563eb')
lim = max(obs.max(), sim.max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'k--', lw=0.6)
axes[0].set_xlabel('Observado (m³/s)'); axes[0].set_ylabel('Predicho (m³/s)')
axes[0].set_title('Predicho vs observado')

axes[1].plot(df_diag.index, df_diag['err'], color='#c2410c', lw=0.4)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('Error (pred − obs)')
axes[1].set_title('Residuos en el tiempo')
plt.tight_layout()

**Patrón típico:** infraestimación en crecidas (errores negativos en los picos) y buen ajuste en estiaje/medio. Es la **firma de los modelos de árbol** — no extrapolan.

## 4 · Ejercicios

1. **Refit=True.** Repite el backtest con `refit=True`. ¿Cuánto tarda? ¿Cambian las métricas?
2. **Distintos horizontes.** Repite con `steps=1`, `7`, `30`. Compara cómo crece el MAE con el horizonte.
3. **Quantile.** Reajusta LightGBM con `objective='quantile', alpha=0.9` y compara el error de pico contra el modelo de la media.
4. **Comparar con Sesión 2.** Toma las predicciones SARIMAX del notebook 2.4 (mensuales) y compara, agregando aquí a mensual.
5. **Reto.** Implementa **conformal prediction**: tras backtest, calcula los cuantiles empíricos del error de validación; úsalos para construir un IC del 90% sobre el test.